In [ ]:
!pip install vllm transformers accelerate pandas tqdm

In [ ]:
SYSTEM_PROMPT = """
You are an expert in customer experience analysis, Net Promoter Score (NPS),
sentiment analysis, and cross-cultural interpretation of reviews.

Your task:
1. Infer sentiment on a scale from 1 to 5
2. Classify the customer as:
   - Promoter
   - Passive
   - Detractor

Important:
- Cultural norms influence how customers express satisfaction.
- Customers from high Power Distance or high Uncertainty Avoidance cultures
  may express dissatisfaction indirectly.
- Customers from indulgent cultures may exaggerate positivity.

Examples:
- Japan (high UAI): neutral wording may indicate dissatisfaction.
- USA (high IVR): positive language is more explicit.
- Germany (high UAI): criticism is direct and precise.

Use this awareness while classifying.
Return ONLY valid JSON.
"""

In [ ]:
def build_user_prompt(review_body, country, product_category):
    return f"""
Review Text:
\"\"\"{review_body}\"\"\"

Country:
{country}

Product Category:
{product_category}

Tasks:
1. Assign a sentiment score from 1 (very negative) to 5 (very positive).
2. Assign one NPS category: Promoter, Passive, or Detractor.

Output format (JSON only):
{{
  "sentiment_score": <1-5 integer>,
  "nps_category": "<Promoter | Passive | Detractor>"
}}
"""

In [ ]:
from vllm import LLM, SamplingParams
import torch

MODEL_NAME = "google/gemma-3-27b-it"

llm = LLM(
    model=MODEL_NAME,
    tensor_parallel_size=1,        # ✅ single H100
    dtype="bfloat16",
    gpu_memory_utilization=0.90,   # safe margin
    max_model_len=4096
)

sampling_params = SamplingParams(
    temperature=0.0,
    top_p=1.0,
    max_tokens=256
)

In [ ]:
import json

def run_llm_batch(rows):
    prompts = []

    for _, row in rows.iterrows():
        user_prompt = build_user_prompt(
            review_body=row["review_body_en"],
            country=row["country"],
            product_category=row["product_category"]
        )

        prompts.append(
            f"<system>\n{SYSTEM_PROMPT}\n</system>\n<user>\n{user_prompt}\n</user>"
        )

    outputs = llm.generate(prompts, sampling_params)

    results = []
    for out in outputs:
        text = out.outputs[0].text.strip()
        try:
            parsed = json.loads(text)
        except Exception:
            parsed = {
                "sentiment_score": None,
                "nps_category": "UNKNOWN"
            }
        results.append(parsed)

    return results

In [ ]:
import pandas as pd
from tqdm import tqdm
import os

INPUT_CSV = "nps_raw.csv"
OUTPUT_CSV = "nps_with_gemma27b.csv"

BATCH_SIZE = 32
SAVE_EVERY = 500

df = pd.read_csv(INPUT_CSV)

if "sentiment_score_gemma27b" not in df.columns:
    df["sentiment_score_gemma27b"] = None
    df["nps_category_gemma27b"] = None

start_idx = df["sentiment_score_gemma27b"].isna().idxmax()

print(f"Starting inference from row {start_idx}")

for i in tqdm(range(start_idx, len(df), BATCH_SIZE)):
    batch = df.iloc[i:i+BATCH_SIZE]

    results = run_llm_batch(batch)

    for idx, res in zip(batch.index, results):
        df.at[idx, "sentiment_score_gemma27b"] = res["sentiment_score"]
        df.at[idx, "nps_category_gemma27b"] = res["nps_category"]

    if i % SAVE_EVERY == 0:
        df.to_csv(OUTPUT_CSV, index=False)
        print(f"Checkpoint saved at row {i}")

df.to_csv(OUTPUT_CSV, index=False)
print("✅ Gemma-27B inference completed")